# Final Presentation - Full Platform Layers

This notebook is the final presentation checklist after the PR1 and PR2 data notebooks. It verifies the remaining mandatory layers: Delta Lake, Structured Streaming, Airflow, MLflow registry, BentoML serving, Prometheus monitoring, Grafana dashboards, tests, and the 10x API load test.

## 1. Delta Lake Tables

Production command:

```bash
python -m spark_jobs.build_delta_lakehouse --year 2024 --month 1
```

The job writes Delta transaction logs for Bronze weather, Bronze BTS, Silver flight-weather labels, and Gold training features.

## 2. Spark Structured Streaming

Production command:

```bash
python -m spark_jobs.stream_kafka_weather_to_delta --trigger-once
```

This uses Spark `readStream` from Kafka and `writeStream` into a Delta sink with a MinIO checkpoint.

## 3. MLflow Registry

Production command:

```bash
python -m ml.train_register_model
```

The final model uses Spark MLlib Logistic Regression, chronological evaluation, and positive-class weighting. Its exported coefficient model is logged and registered with staging and production aliases.

In [1]:
import requests

services = {
    "BentoML API": "http://aviation-api:3000/metrics",
    "Prometheus": "http://prometheus:9090/-/healthy",
    "Grafana": "http://grafana:3000/api/health",
    "MLflow": "http://mlflow:5000/health",
}
for name, url in services.items():
    response = requests.get(url, timeout=5)
    print(f"{name:12} status={response.status_code} url={url}")

BentoML API  status=200 url=http://aviation-api:3000/metrics
Prometheus   status=200 url=http://prometheus:9090/-/healthy
Grafana      status=200 url=http://grafana:3000/api/health
MLflow       status=200 url=http://mlflow:5000/health


## 4. BentoML Prediction

In [2]:
sample = {
    "scheduled_departure_hour_local": 16.0, "distance_miles": 1090.0,
    "day_of_week": 2.0, "day_of_month": 15.0, "temperature_c_avg": 5.5,
    "wind_speed_kts_max": 24.0, "wind_gust_kts_max": 31.0,
    "precipitation_mm_sum": 6.0, "surface_pressure_pa_avg": 100900.0,
    "total_cloud_cover_avg": 0.75, "cape_j_kg_max": 80.0,
}
response = requests.post("http://aviation-api:3000/predict", json={"features": sample}, timeout=10)
response.raise_for_status()
response.json()

{'prediction': 1,
 'disruption_probability': 0.717981,
 'risk_band': 'high',
 'model_name': 'aviation-disruption-balanced-logistic'}

## 5. Presentation URLs

- Airflow: <http://localhost:8088>
- MLflow: <http://localhost:5000>
- BentoML API: <http://localhost:3000>
- Prometheus: <http://localhost:9090>
- Grafana: <http://localhost:3001>

Run `python3 api/load_test.py --requests 100 --concurrency 10` before opening Grafana so the dashboard contains live traffic.